In [1]:
%load_ext autoreload
%autoreload 2

In [2]:
# models generation directory: LLM4DC/CoT.response
# predicted dataset by model:  LLM4DC/CoT.response/{model}/datasets_llm
# predicted workflow by model:  LLM4DC/CoT.response/{model}/recipes_llm
# predicted operations by model:  LLM4DC/CoT.response/{model}/operation
# logging:  LLM4DC/CoT.response/logging

# all the sample tables are under: LLM4DC/datasets
# query: 1-30 [menu]: LLM4DC/datasets/menu_datasets
# clean table (ground truth) LLM4DC/datasets/menu_datasets/clean_tables/
# clean workflow (silver ground truth):  LLM4DC/datasets/menu_datasets/workflows

# query: 31-61 [chicago]: LLM4DC/datasets/CFI_datasets
# clean table (ground truth): LLM4DC/datasets/CFI_datasets/cleaned_tables/
# clean workflow (silver ground truth):  LLM4DC/datasets/CFI_datasets/workflows

# query: 62-91 [ppp]:LLM4DC/datasets/ppp_datasets
# clean table (ground truth): LLM4DC/datasets/ppp_datasets/cleaned_tables/
# clean workflow (silver ground truth):  LLM4DC/datasets/ppp_datasets/workflows

# query: 92-110 [dish]: LLM4DC/datasets/dish_datasets
# clean table (ground truth): LLM4DC/datasets/dish_datasets/cleaned_tables/
# clean workflow (silver ground truth):  LLM4DC/datasets/dish_datasets/workflows

# query: 111-126 [flights]: LLM4DC/datasets/flights
# clean table (ground truth): LLM4DC/datasets/flights/cleaned_tables/flights_data_p{query_id}.csv
# clean workflow (silver ground truth):  LLM4DC/datasets/flights/workflows/flights_p{query_id}.json

# query: 127- 154 [hospital]: LLM4DC/datasets/hospital
# clean table (ground truth): LLM4DC/datasets/hospital/clean_tables/
# clean workflow (silver ground truth):  LLM4DC/datasets/hospital/workflows

In [2]:
import re

In [3]:
import json

In [4]:
import ast
# LLM-based history update solution
import importlib.util
import inspect
from typing import List
import requests
import json
import re
import difflib
from collections import Counter
# from spellchecker import SpellChecker
from datetime import datetime
import pandas as pd
import ast
import random
import logging 
# from history_update_problem.call_or import export_rows
from call_or import *
from evaluation import *

In [5]:
models = ['llama3.1',  'mistral', 'gemma2','deepseek-r1']

# Worflow eval

In [6]:
def eval_workflows(pp_id, gt_wf_fp, pred_wf_fp):
    # print(gt_wf_fp)
    gt_ops_list = parse_recipe(pp_id, recipe=gt_wf_fp)
    pred_ops_list = parse_recipe(pp_id, recipe=pred_wf_fp)
    return {'pp_id': pp_id, 'gt_ops': gt_ops_list[pp_id], 'pred_ops': pred_ops_list[pp_id]}
    # print(gt_ops_list)
    # print(pred_ops_list)

answer_gt_path = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_gt.json'
answer_preds_llama = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_llama3.1.json'


# eval_answer_results = eval_answers(answer_gt_path, answer_preds_llama)
# model = models[0]
wf_gt_folder = '/projects/bces/lanl2/LLM4DC/datasets'

query_contents = pd.read_csv('/projects/bces/lanl2/LLM4DC/purposes/all_purposes.csv')
    
ops_result = {}
for model in models[:3]:
    ops_list = []

    wf_pred_folder = f'/projects/bces/lanl2/LLM4DC/CoT.response/{model}/recipes_llm'
    for query_id in range(155):
        row = query_contents[query_contents['ID'] == query_id]
        if len(row) == 0:
            continue
        # if model == 'llama3.1':
        if query_id >126:
            #TODO: what's the point to have the target_path here? 
            target_path = f'{wf_gt_folder}/hospital/clean_tables/hos_pp{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/hospital/workflows/hos_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_hos_test_p{query_id}.json"
        elif query_id >= 111 and query_id <=126:
            target_path = f'{wf_gt_folder}/flights/cleaned_tables/flights_data_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/flights/workflows/flights_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_flights_test_p{query_id}.json"
        elif query_id >= 92 and query_id <=110:
            target_path = f'{wf_gt_folder}/dish_datasets/cleaned_tables/dish_sample_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/dish_datasets/workflows/dish_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_dish_test_p{query_id}.json"
        elif query_id >= 62 and query_id <= 91:
            target_path = f'{wf_gt_folder}/ppp_datasets/cleaned_tables/ppp_sample_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/ppp_datasets/workflows/ppp_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_ppp_test_p{query_id}.json"
        elif query_id >= 31 and query_id <= 61:
            target_path = f'{wf_gt_folder}/CFI_datasets/cleaned_tables/chi_sample_p{query_id}.csv'
            wf_gt_fp = f"{wf_gt_folder}/CFI_datasets/workflows/chi_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_chi_test_p{query_id}.json"

        elif query_id <31:
            target_path = f'{wf_gt_folder}/menu_datasets/clean_tables/menu_sample_p{query_id}'
            wf_gt_fp = f"{wf_gt_folder}/menu_datasets/workflows/menu_sample_p{query_id}.json"
            wf_pred_fp = f"{wf_pred_folder}/{model}_menu_test_p{query_id}.json"
        if wf_gt_fp and wf_pred_fp:
            ops_list.append(eval_workflows(query_id, wf_gt_fp, wf_pred_fp))
    ops_result[model] = pd.DataFrame(ops_list)
# ops_df = pd.DataFrame(ops_list)

In [7]:
def parse_mean(df, col_name):
    total_wf = df.describe().loc['mean']
    # total_wf.columns = [col_name] #rename(columns={"mean": col_name})
    return total_wf

In [8]:
wf_length = []
total_wf = []
print(len(ops_result))
ppp_wf, dish_wf, menu_wf,chi_wf,hos_wf, flights_wf = [],[],[],[],[],[]
col_name = []
output_wf =  []
for key, value in ops_result.items():
    model = key
    ops_df = value
    ops_df['gt_ops_length'] = ops_df['gt_ops'].apply(len)
    ops_df['pred_ops_length'] = ops_df['pred_ops'].apply(len)
    ops_df['gt_ops_set_length'] = ops_df['gt_ops'].apply(lambda x: len(set(x)))
    ops_df['pred_ops_set_length'] = ops_df['pred_ops'].apply(lambda x: len(set(x)))

    ops_length_desc = ops_df.describe().loc['mean']
    ops_length_desc.columns = [model]
    wf_length.append(ops_length_desc)


    workflow_results = calculate_operation_metrics(ops_df['gt_ops'], ops_df['pred_ops'])
    workflow_results['pp_id'] = ops_df['pp_id']
    # print(workflow_results[workflow_results['pp_id'] == 127])
    workflow_results.set_index('pp_id', inplace=True)
    ops_df.set_index('pp_id', inplace=True)
    total_wf.append(parse_mean(workflow_results, f'total__{model}'))
    col_name.append(f'total__{model}')
    
    
    hos_results = workflow_results.loc[127:155]
       
    # hos_results = workflow_results[workflow_results['pp_id'] >= 127]
    total_wf.append(parse_mean(hos_results, f'hos__{model}'))
    hos_ops = ops_df.loc[127:155]
    wf_length.append(parse_mean(hos_ops, f'hos__{model}'))
    col_name.append(f'hos__{model}')


    # flights_results = workflow_results[workflow_results['pp_id'] >= 111][workflow_results['pp_id'] <=126]
    flights_results = workflow_results.loc[111:127]
    flights_ops = ops_df.loc[111:127]
    total_wf.append(parse_mean(flights_results, f'flights__{model}'))
    wf_length.append(parse_mean(flights_ops, f'flights_{model}'))
    col_name.append(f'flights__{model}')

    # ppp_results = workflow_results[workflow_results['pp_id'] >= 62][workflow_results['pp_id'] <=91]
    ppp_results = workflow_results.loc[62:91]
    ppp_ops = ops_df.loc[62:91] 
    wf_length.append(parse_mean(ppp_ops, f'ppp_{model}'))
    total_wf.append(parse_mean(ppp_results, f'ppp__{model}'))
    col_name.append(f'ppp__{model}')

    # dish_results = workflow_results[workflow_results['pp_id'] >= 92][workflow_results['pp_id'] <= 110]
    dish_results = workflow_results.loc[92:111]
    dish_ops = ops_df.loc[92:111] 
    wf_length.append(parse_mean(dish_ops, f'dish_{model}'))
    total_wf.append(parse_mean(dish_results, f'dish__{model}'))
    col_name.append(f'dish__{model}')

    # menu_results = workflow_results[workflow_results['pp_id'] < 31]
    menu_results = workflow_results.loc[:31]
    menu_ops = ops_df.loc[:31] 
    wf_length.append(parse_mean(menu_ops, f'menu_{model}'))
    total_wf.append(parse_mean(menu_results, f'menu__{model}'))
    col_name.append(f'menu__{model}')
    # chi_results = workflow_results[workflow_results['pp_id'] >= 31][workflow_results['pp_id'] <=61]
    chi_results = workflow_results.loc[31:62]
    chi_ops = ops_df.loc[92:111] 
    wf_length.append(parse_mean(chi_ops, f'chi_{model}'))
    total_wf.append(parse_mean(chi_results, f'chi__{model}'))
    col_name.append(f'chi__{model}')
wf_perf = pd.concat(total_wf, axis=1)
wf_perf.columns = col_name
# print(wf_perf)
wf_perf = wf_perf.transpose()
wf_length = pd.concat(wf_length, axis=1)
wf_length.columns = col_name
wf_length = wf_length.transpose()
wf_perf = wf_perf.reset_index()

3


In [9]:
wf_length

,pp_id,gt_ops_length,pred_ops_length,gt_ops_set_length,pred_ops_set_length
total__llama3.1,76.739437,7.492958,4.323944,3.563380,1.943662
hos__llama3.1,NaN,10.178571,4.714286,3.964286,1.785714
flights__llama3.1,NaN,8.647059,3.588235,3.882353,1.529412
ppp__llama3.1,NaN,6.090909,4.181818,3.500000,2.136364
dish__llama3.1,NaN,5.470588,3.882353,3.764706,2.294118
menu__llama3.1,NaN,6.903226,4.580645,2.870968,2.064516
chi__llama3.1,NaN,5.470588,3.882353,3.764706,2.294118
total__mistral,76.739437,7.492958,3.429577,3.563380,1.838028
hos__mistral,NaN,10.178571,4.321429,3.964286,1.964286
flights__mistral,NaN,8.647059,3.294118,3.882353,1.647059


In [10]:
wf_length.to_csv('workflow_length.csv')

In [11]:
wf_perf

,index,accuracy,precision,recall,f1
0,total__llama3.1,0.098592,0.933099,0.526526,0.644679
1,hos__llama3.1,0.000000,1.000000,0.458333,0.618707
2,flights__llama3.1,0.000000,0.970588,0.383333,0.534874
3,ppp__llama3.1,0.272727,0.958333,0.665152,0.755880
4,dish__llama3.1,0.117647,0.867647,0.514706,0.614426
5,menu__llama3.1,0.193548,0.870968,0.623118,0.701101
6,chi__llama3.1,0.064516,0.940860,0.512366,0.638249
7,total__mistral,0.049296,0.765610,0.409624,0.506539
8,hos__mistral,0.000000,0.922619,0.453571,0.591412
9,flights__mistral,0.000000,0.852941,0.357843,0.492717


In [12]:
wf_perf.to_csv('workflow_results_llama_mistral_gemma2.csv')

# Dataset Eval

In [27]:
# def retrieve_tg_cols(tg_cols_fp="target_columns_list.csv"):
#     id_tg_cols = {}
#     tg_df = pd.read_csv(tg_cols_fp)
#     result_dict = tg_df.set_index('ID')['tg_columns'].to_dict()
#     return result_dict

In [15]:
result_dict = retrieve_tg_cols("/projects/bces/lanl2/LLM4DC/evaluation/target_column_list.csv")
print(result_dict)
# model = "llama3.1"
# model = "mistral"
# model = "gemma2"
# model = "dirty"
data_gt_folder = "/projects/bces/lanl2/LLM4DC/datasets"

for model in models[:3] + ["dirty"]:
    ratio_list = []
    for query_id in range(155):
        # print(query_id)
        tg_cols = result_dict.get(query_id)
        if tg_cols:
            if model=="dirty":
                if query_id >126:
                    target_path = f'{data_gt_folder}/hospital/clean_tables/hos_pp{query_id}.csv'
                    table_preds_path = f'/projects/bces/lanl2/LLM4DC/datasets/hospital/hos_data_p{query_id}.csv'
                elif query_id >= 111 and query_id <=126:
                    target_path = f'{data_gt_folder}/flights/cleaned_tables/flights_data_p{query_id}.csv'
                    table_preds_path = f'/projects/bces/lanl2/LLM4DC/datasets/flights/flights_data_p{query_id}.csv'
                    # target_path = None
                    # table_preds_path = None
                elif query_id >= 92 and query_id <=110:
                    target_path = f'{data_gt_folder}/dish_datasets/cleaned_tables/dish_sample_p{query_id}.csv'
                    table_preds_path = f'/projects/bces/lanl2/LLM4DC/datasets/dish_datasets/dish_data_p{query_id}.csv'
                elif query_id >= 62 and query_id <= 91:
                    target_path = f'{data_gt_folder}/ppp_datasets/cleaned_tables/ppp_sample_p{query_id}.csv'
                    table_preds_path =  f'/projects/bces/lanl2/LLM4DC/datasets/ppp_datasets/ppp_data_p{query_id}.csv'
                elif query_id >= 31 and query_id <= 61:
                    target_path = f'{data_gt_folder}/CFI_datasets/cleaned_tables/chi_sample_p{query_id}.csv'
                    table_preds_path = f'/projects/bces/lanl2/LLM4DC/datasets/CFI_datasets/chi_food_data_p{query_id}.csv'
                elif query_id <31:
                    target_path = f'{data_gt_folder}/menu_datasets/clean_tables/menu_sample_p{query_id}.csv'
                    table_preds_path = f'/projects/bces/lanl2/LLM4DC/datasets/menu_datasets/menu_p{query_id}.csv'
            else:
                llm_folder = f"CoT.response/{model}/datasets_llm"
                data_gt_folder = "/projects/bces/lanl2/LLM4DC/datasets"
                pred_fp = f'/projects/bces/lanl2/LLM4DC/{llm_folder}'
                if query_id >126:
                    target_path = f'{data_gt_folder}/hospital/clean_tables/hos_pp{query_id}.csv'
                    table_preds_path = f'{pred_fp}/{model}_hos_test_p{query_id}.csv'
                elif query_id >= 111 and query_id <=126:
                    target_path = f'{data_gt_folder}/flights/cleaned_tables/flights_data_p{query_id}.csv'
                    table_preds_path = f'{pred_fp}/{model}_flights_test_p{query_id}.csv'
                elif query_id >= 92 and query_id <=110:
                    target_path = f'{data_gt_folder}/dish_datasets/cleaned_tables/dish_sample_p{query_id}.csv'
                    table_preds_path = f'{pred_fp}/{model}_dish_test_p{query_id}.csv'
                elif query_id >= 62 and query_id <= 91:
                    target_path = f'{data_gt_folder}/ppp_datasets/cleaned_tables/ppp_sample_p{query_id}.csv'
                    table_preds_path = f'{pred_fp}/{model}_ppp_test_p{query_id}.csv'
                elif query_id >= 31 and query_id <= 61:
                    target_path = f'{data_gt_folder}/CFI_datasets/cleaned_tables/chi_sample_p{query_id}.csv'
                    table_preds_path = f'{pred_fp}/{model}_chi_test_p{query_id}.csv'
                elif query_id <31:
                    target_path = f'{data_gt_folder}/menu_datasets/clean_tables/menu_sample_p{query_id}.csv'
                    table_preds_path = f'{pred_fp}/{model}_menu_test_p{query_id}.csv'
            if target_path and table_preds_path:
                gt_df = pd.read_csv(target_path)
                preds_df = pd.read_csv(table_preds_path)
                # print(gt_df.head(5), preds_df.head(5))
                res = average_match_ratio(gt_df, preds_df, tg_cols)
                ratio_list.append({'pp_id': query_id, 'ratio': res})
    dt_result = pd.DataFrame(ratio_list)
    dt_result.to_csv(f'evaluation/{model}_table_result.csv')

{1: 'page_count', 2: 'page_count', 3: 'event', 4: 'event', 5: 'event', 6: 'venue', 7: 'occasion', 8: 'occasion', 9: 'page_count, dish_count', 10: 'dish_count', 11: 'page_count, dish_count', 12: 'page_count, location', 13: 'sponsor, currency', 14: 'sponsor, dish_count', 15: 'sponsor, event', 16: 'sponsor, event', 17: 'sponsor, event', 18: 'sponsor, event', 19: 'page_count, venue', 20: 'sponsor', 21: 'event', 22: 'occasion', 23: 'venue, dish_count', 24: 'status', 25: 'sponsor, currency', 26: 'date', 27: 'date, page_count, dish_count', 28: 'page_count, venue', 29: 'occasion', 30: 'currency', 31: 'Risk', 32: 'Results', 33: 'Facility Type', 34: 'Facility Type', 35: 'Inspection Type', 36: 'DBA Name, Results', 37: 'DBA Name, Results', 38: 'Facility Type, Risk', 39: 'Facility Type, Risk', 40: 'Facility Type, Risk', 41: 'Facility Type, Risk', 42: 'Facility Type, Risk', 43: 'Facility Type, Results', 44: 'Results', 45: 'Facility Type, Inspection ID', 46: 'Risk', 47: 'Facility Type, Results', 48: 

In [16]:
dt_result

,pp_id,ratio
0,1,0.300000
1,2,1.000000
2,3,0.565217
3,4,0.869565
4,5,0.152174
...,...,...
137,150,0.450000
138,151,0.450000
139,152,0.300000
140,153,0.775000


In [17]:
total_tab = []
col_name = []
for model in models[:3] + ["dirty"]:
    print(model)
    
    table_results = pd.read_csv(f'evaluation/{model}_table_result.csv')
    table_results.set_index('pp_id', inplace=True)

    total_tab.append(parse_mean(table_results, f'total__{model}'))
    col_name.append(f'total__{model}')
    
    
    hos_results = table_results.loc[127:155]
    hos_results.reset_index()
    # hos_results = workflow_results[workflow_results['pp_id'] >= 127]
    total_tab.append(parse_mean(hos_results, f'hos__{model}'))
    
    col_name.append(f'hos__{model}')


    # flights_results = workflow_results[workflow_results['pp_id'] >= 111][workflow_results['pp_id'] <=126]
    flights_results = table_results.loc[111:127]
    flights_results.reset_index()
    total_tab.append(parse_mean(flights_results, f'flights__{model}'))
    col_name.append(f'flights__{model}')

    # ppp_results = workflow_results[workflow_results['pp_id'] >= 62][workflow_results['pp_id'] <=91]
    ppp_results = table_results.loc[62:91]
    ppp_results.reset_index()
    total_tab.append(parse_mean(ppp_results, f'ppp__{model}'))
    col_name.append(f'ppp__{model}')

    # dish_results = workflow_results[workflow_results['pp_id'] >= 92][workflow_results['pp_id'] <= 110]
    dish_results = table_results.loc[92:111]
    dish_results.reset_index()
    total_tab.append(parse_mean(dish_results, f'dish__{model}'))
    col_name.append(f'dish__{model}')

    # menu_results = workflow_results[workflow_results['pp_id'] < 31]
    menu_results = table_results.loc[:31]
    menu_results = menu_results.reset_index()
    total_tab.append(parse_mean(menu_results, f'menu__{model}'))
    col_name.append(f'menu__{model}')
    # chi_results = workflow_results[workflow_results['pp_id'] >= 31][workflow_results['pp_id'] <=61]
    chi_results = table_results.loc[31:62]
    chi_results = chi_results.reset_index()
    total_tab.append(parse_mean(chi_results, f'chi__{model}'))
    col_name.append(f'chi__{model}')
tab_perf = pd.concat(total_tab, axis=1)
tab_perf.columns = col_name
# print(wf_perf)
tab_perf = tab_perf.transpose()
tab_perf.to_csv('table_column_ratio_results_llama_mistral_gemma2.csv')

llama3.1
mistral
gemma2
dirty


In [120]:
dt_result.head()

,pp_id,ratio
0,1,0.30
1,2,1.00
2,3,0.26
3,4,0.40
4,5,0.07


In [18]:
stat = dt_result['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

0.42616081449511906
0.285582545474569
0.0
1.0
0.2
0.4557692307692308
0.5958333333333333


In [122]:
hos_dt_results = dt_result[dt_result['pp_id'] >= 127]
stat = hos_dt_results['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

0.4532738095238095
0.13935734489692167
0.21666666666666665
0.775
0.35625
0.45
0.5270833333333333


In [123]:
flights_dt_results = dt_result[dt_result['pp_id'] >= 111][dt_result['pp_id'] <=126]
stat = flights_dt_results['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

0.04666666666666667
0.08944271909999159
0.0
0.34
0.0
0.0
0.06833333333333333


/tmp/ipykernel_4172012/1502436861.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  flights_dt_results = dt_result[dt_result['pp_id'] >= 111][dt_result['pp_id'] <=126]


In [ ]:
ppp_dt_results = dt_result[dt_result['pp_id'] >= 62][dt_result['pp_id'] <=91]
stat = ppp_dt_results['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

In [ ]:
dish_dt_result =dt_result[dt_result['pp_id'] >= 92][dt_result['pp_id'] <=110]
stat = dish_dt_result['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

In [ ]:
chi_dt_result =dt_result[dt_result['pp_id'] >= 31][dt_result['pp_id'] <=61]
stat = chi_dt_result['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

In [ ]:
menu_dt_result =dt_result[dt_result['pp_id'] < 31]

stat = menu_dt_result['ratio'].describe()
for idx in ['mean', 'std',	'min',	'max',	'25%',	'50%',	'75%']:
    print(stat.loc[idx])

## table eval ttest

# answer eval

In [19]:
par_folder = '/projects/bces/lanl2/LLM4DC'
datafile_path = f'{par_folder}/evaluation/answer_1-154_gt.json'
data = []
with open(datafile_path, 'r') as f:
    for l in f:
        data.append(json.loads(l))

In [20]:
from bert_score import score

/projects/bces/lanl2/autodc/lib/python3.11/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [21]:
import json
from typing import Union, List, Dict, Any
from difflib import SequenceMatcher
from math import isclose

# Define a utility to convert JSON strings to Python objects
def parse_input(answer: Union[str, float, List, Dict]) -> Any:
    if isinstance(answer, str):
        try:
            # Try to parse JSON strings into Python objects
            return json.loads(answer)
        except json.JSONDecodeError:
            return answer.lower().strip()  # Normalize strings for comparison
    elif isinstance(answer, float):
        return round(answer, 2)  # Round floats to two decimal places if needed
    return answer  # If already in desired format

# Calculate exact match accuracy
def accuracy_metric(gt: Any, pred: Any) -> float:
    return 1.0 if gt == pred else 0.0

# Calculate precision, recall, and F1 for lists (assuming items are unique)
def precision_recall_f1(gt: List, pred: List) -> Dict[str, float]:
    gt_set, pred_set = set(gt), set(pred)
    true_positives = len(gt_set & pred_set)
    precision = true_positives / len(pred_set) if pred_set else 0
    recall = true_positives / len(gt_set) if gt_set else 0
    f1_score = 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0
    return {"precision": precision, "recall": recall, "f1": f1_score}

# Calculate semantic distance for string answers using sequence matching
def semantic_similarity(gt: str, pred: str) -> float:
    return SequenceMatcher(None, gt, pred).ratio()  # Returns a ratio between 0 and 1

# Evaluate an answer based on the ground truth
def calculate_answer_metrics(gt: Any, pred: Any) -> Dict[str, float]:
    # Parse inputs
    gt, pred = parse_input(gt), parse_input(pred)
    
    # Initialize results
    results = {"accuracy": 0, "semantic_similarity": 0, "precision": 0, "recall":0, "f1":0}
    
    # Check type and apply appropriate metrics
    if isinstance(gt, float) and isinstance(pred, float):
        results["accuracy"] = 1.0 if isclose(gt, pred, rel_tol=1e-2) else 0.0  # Accuracy for floats with tolerance
        precision, recall = results["accuracy"], results["accuracy"]
        results.update({"precision": precision, "recall": recall, "f1": 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0})
        # results['bertscore'] = score([f"{pred}"], [f"{gt}"], lang='en', verbose=True)
        results["semantic_similarity"] = semantic_similarity(str(gt), str(pred))
    elif isinstance(gt, int) and isinstance(pred, int):
        results["accuracy"] = 1.0 if isclose(gt, pred, rel_tol=1e-2) else 0.0  # Accuracy for floats with tolerance
        precision, recall = results["accuracy"], results["accuracy"]
        results.update({"precision": precision, "recall": recall, "f1": 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0})
        # results['bertscore'] = score([f"{pred}"], [f"{gt}"], lang='en', verbose=True)
        results["semantic_similarity"] = semantic_similarity(str(gt), str(pred))


    elif isinstance(gt, str) and isinstance(pred, str):
        results["accuracy"] = accuracy_metric(gt.lower(), pred.lower())
        precision, recall = results["accuracy"], results["accuracy"]
        results.update({"precision": precision, "recall": recall, "f1": 2 * precision * recall / (precision + recall) if precision + recall > 0 else 0})
        # results['bertscore'] = score([pred], [gt], lang='en', verbose=True)
        results["semantic_similarity"] = semantic_similarity(gt, pred)

    
    elif isinstance(gt, list) and isinstance(pred, list):
        if type(gt[0]) == str:
            gt = [x.lower() for x in gt]
            if len(pred) > 0:
                if type(pred[0]) == str:
                    pred = [x.lower() for x in pred]
        metrics = precision_recall_f1(gt, pred)
        results.update(metrics)
        results["accuracy"] = accuracy_metric(gt, pred)
        # results['bertscore'] = score([f"{pred}"], [f"{gt}"], lang='en', verbose=True)
        similarity = semantic_similarity(f"{gt}", f"{pred}")
        results["semantic_similarity"] = similarity

    
    elif isinstance(gt, dict) and isinstance(pred, dict):
        gt = {key.lower(): value for key, value in gt.items()}
        pred = {key.lower(): value for key, value in pred.items()}
        gt_keys, pred_keys = list(gt.keys()), list(pred.keys())
        if type(gt[gt_keys[0]]) == dict:
            precision_recall_f1_results = []
            for k in gt_keys:
                if k in pred.keys():
                    pred_input = pred[k].values()
                else:
                    pred_input = []
                precision_recall_f1_results.append(precision_recall_f1(gt[k].values(), pred_input))
            precision = sum([x['precision'] for x in precision_recall_f1_results])/len([x['precision'] for x in precision_recall_f1_results])
            recall = sum([x['recall'] for x in precision_recall_f1_results])/len([x['recall'] for x in precision_recall_f1_results])
            f1 =sum([x['f1'] for x in precision_recall_f1_results])/len([x['f1'] for x in precision_recall_f1_results])
            results.update({'precision': precision, 'recall':recall, 'f1': f1})  # Precision/Recall on keys

        elif type(gt[gt_keys[0]]) == list:
            precision_recall_f1_results = []
            for k in gt_keys:
                if k in pred:
                    pred_input = pred[k]
                else: 
                    pred_input = []
                if type(gt[k][0]) ==str:
                    gt_input = [x.lower() for x in gt[k]]
                    if len(pred_input) > 0:
                        if type(pred_input[0]) == str:
                            pred_input = [x.lower() for x in pred_input]    
                else:
                    gt_input = gt[k]
                precision_recall_f1_results.append(precision_recall_f1(gt[k], pred_input)) 
            precision = sum([x['precision'] for x in precision_recall_f1_results])/len([x['precision'] for x in precision_recall_f1_results])
            recall = sum([x['recall'] for x in precision_recall_f1_results])/len([x['recall'] for x in precision_recall_f1_results])
            f1 =sum([x['f1'] for x in precision_recall_f1_results])/len([x['f1'] for x in precision_recall_f1_results])
            results.update({'precision': precision, 'recall':recall, 'f1': f1})  # Precision/Recall on keys
        

        results["accuracy"] = accuracy_metric(gt, pred)
        # Check semantic similarity for each key-value pair
        similarity = [semantic_similarity(str(gt[k]), str(pred.get(k, ""))) for k in gt_keys]
        results["semantic_similarity"] = sum(similarity) / len(similarity) if similarity else 0
        
        # results['bertscore'] = score([f"{pred}"], [f"{gt}"], lang='en', verbose=True)
    p, r, f1 = score([f"{pred}"], [f"{gt}"], lang='en', verbose=True)
    results.update({'bertscore_p': p.detach().cpu().tolist(),
    'bertscore_r': r.detach().cpu().tolist(),
    'bertscore_f1': f1.detach().cpu().tolist()})
    # print(results)


    return results

In [42]:
test_json = "{\"Zip\":{\"0\":96701,\"1\":96704,\"2\":96707,\"3\":96708,\"4\":96749,\"5\":96750,\"6\":96754,\"7\":96791,\"8\":96813,\"9\":96814,\"10\":96815,\"11\":96816,\"12\":96817,\"13\":96821,\"14\":96825,\"15\":96826},\"LoanCount\":{\"0\":1,\"1\":1,\"2\":4,\"3\":1,\"4\":1,\"5\":1,\"6\":1,\"7\":1,\"8\":1,\"9\":1,\"10\":1,\"11\":2,\"12\":1,\"13\":1,\"14\":1,\"15\":1}}"

In [43]:
test_json = parse_input(test_json)

In [ ]:
test_json.keys()
test_json.get('Zip').values()

In [22]:
# LLM-based history update solution
import importlib.util
import inspect
from typing import List
import requests
import json
import re
import difflib
from collections import Counter
# from spellchecker import SpellChecker
from datetime import datetime
import pandas as pd
import ast
import random
import logging 
# from history_update_problem.call_or import export_rows
from call_or import *

# from evaluation.data_compare import calculate_answer_metrics


def load_answer_dataset(datafile_path):
    """
    load json file, each line is a json dictionary

    datafile_path: str
    return: data:  list_of_dictionary
    """
    data = []
    with open(datafile_path, 'r') as f:
        for l in f:
            data.append(json.loads(l))
    return data

def eval_answers(answer_gt_path, answer_preds_llama):
    answer_gt = load_answer_dataset(answer_gt_path)
    answer_gt = pd.DataFrame(answer_gt)
    answer_preds_llama = load_answer_dataset(answer_preds_llama)
    answer_preds_llama = pd.DataFrame(answer_preds_llama)
    answer_compare = answer_gt.merge(answer_preds_llama[['pp_id', 'answer']], on='pp_id', how='left', suffixes=('_gt', '_preds'))

    results = []
    for i, row in answer_compare.iterrows():
        
        gt = row['answer_gt']
        preds = row['answer_preds']
        
        single_result = calculate_answer_metrics(gt, preds)
        
        single_result['pp_id'] = row['pp_id']
        results.append(single_result)
        # break
    return pd.DataFrame(results)

 
    

In [23]:
# @title single eexample
answer_gt_path = 'evaluation/answer_1-154_gt.json'
answer_gt = load_answer_dataset(answer_gt_path)
answer_gt = pd.DataFrame(answer_gt)
answer_dirty = 'evaluation/answer_1-154_dirty.json'
answer_dirty = load_answer_dataset(answer_dirty)
answer_dirty = pd.DataFrame(answer_dirty)
answer_preds_llama = 'evaluation/answer_1-154_llama3.1.json'
answer_preds_llama = load_answer_dataset(answer_preds_llama)
answer_preds_llama = pd.DataFrame(answer_preds_llama)
answer_compare = answer_gt.merge(answer_preds_llama[['pp_id', 'answer']], on='pp_id', how='left', suffixes=('_gt', '_preds'))
# answer_compare = answer_gt.merge(answer_dirty[['pp_id', 'answer']], on='pp_id', how='left', suffixes=('_gt', '_preds'))

results = []
for i, row in answer_compare.iterrows():
    gt = row['answer_gt']
    preds = row['answer_preds']
    single_result = calculate_answer_metrics(gt, preds)
    single_result['pp_id'] = row['pp_id']
    print(gt, type(gt))
    print(preds, type(preds))
    results.append(single_result)
    break

pd.DataFrame(results)

Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:09<00:00,  9.47s/it]


computing greedy matching.


100%|██████████| 1/1 [00:02<00:00,  2.73s/it]

done in 12.44 seconds, 0.08 sentences/sec
22 <class 'int'>
22 <class 'int'>


,accuracy,semantic_similarity,precision,recall,f1,bertscore_p,bertscore_r,bertscore_f1,pp_id
0,1.0,1.0,1.0,1.0,1.0,[1.0000004768371582],[1.0000004768371582],[1.0000004768371582],1


In [24]:
# 'dirty', 
models = ['llama3.1',  'mistral', 'gemma2','deepseek-r1']
for model in models[0:3] + ['dirty']: #['mistral', 'gemma2', 'llama3.1']:
    print(model)
    answer_gt_path = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_gt.json'
    # answer_preds_llama = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_llama3.1.json'
    # model = 'gemma2'
    # model = 'dirty'
    # model = 'llama3.1'
    # model = 'mistral'
    answer_preds = f'/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_{model}.json'
    # answer_preds = f'/projects/bces/lanl2/LLM4DC/evaluation/answer_1-110_dirty.json'
    eval_answer_results = eval_answers(answer_gt_path, answer_preds)
    eval_answer_results['bertscore_p'] = eval_answer_results['bertscore_p'].apply(lambda x: sum(x)/len(x) if len(x) > 0 else None)
    eval_answer_results['bertscore_r'] = eval_answer_results['bertscore_r'].apply(lambda x: sum(x)/len(x) if len(x) > 0 else None)
    eval_answer_results['bertscore_f1'] = eval_answer_results['bertscore_f1'].apply(lambda x: sum(x)/len(x) if len(x) > 0 else None)
    eval_answer_results.to_csv(f'/projects/bces/lanl2/LLM4DC/evaluation/{model}_answer_result.csv')


llama3.1


Some weights of RobertaModel were not initialized from the model checkpoint at roberta-large and are newly initialized: ['roberta.pooler.dense.bias', 'roberta.pooler.dense.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:00<00:00, 89.45it/s]


computing greedy matching.


...

calculating scores...
computing bert embedding.


100%|██████████| 1/1 [00:00<00:00, 52.48it/s]


computing greedy matching.


100%|██████████| 1/1 [00:00<00:00, 849.22it/s]

done in 0.03 seconds, 39.88 sentences/sec


In [25]:
def calculate_eval_stats(eval_answer_results_text):

    total_stat_table = eval_answer_results_text.describe().loc['mean']
    eval_answer_results_text.set_index('pp_id', inplace=True)

    hos_results = eval_answer_results_text.loc[127:155]
    hos_stat_table = hos_results.describe().loc['mean']
    # print('hos', hos_stat_table)

    flights_results = eval_answer_results_text.loc[111:127]
    flights_stat_table = flights_results.describe().loc['mean']

    ppp_results = eval_answer_results_text.loc[62:92] #[eval_answer_results_text['pp_id'] >= 62][eval_answer_results_text['pp_id'] <=91]
    ppp_stat_table = ppp_results.describe().loc['mean']

    dish_results = eval_answer_results_text.loc[92:111] #[eval_answer_results_text['pp_id'] >= 92]
    dish_stat_table = dish_results.describe().loc['mean']
    
    menu_results = eval_answer_results_text.loc[:31] #[eval_answer_results_text['pp_id'] < 31]
    menu_stat_table = menu_results.describe().loc['mean']

    chi_results = eval_answer_results_text.loc[31:62] #[eval_answer_results_text['pp_id'] >= 31][eval_answer_results_text['pp_id'] <=61]
    chi_stat_table = chi_results.describe().loc['mean']


    final_result = pd.concat([total_stat_table, menu_stat_table, dish_stat_table, chi_stat_table, ppp_stat_table, hos_stat_table, flights_stat_table], axis=1)
    final_result = final_result.transpose()
    final_result['data'] = ['Total', 'Menu', 'Dish', 'CFI','PPP', 'Hospital', 'Flights' ]
    return final_result

In [26]:
total_stat_df = []
for model in models[:3] + ['dirty']: #, 'llama3.1', 'mistral', 'gemma2']:
    eval_answer_results = pd.read_csv(f'/projects/bces/lanl2/LLM4DC/evaluation/{model}_answer_result.csv')
    # evaL_answer_results_flag = eval_answer_results.merge(numeric_pp_tag, on='pp_id', how='left', suffixes=('', '_numeric'))
    # eval_answer_results_text = evaL_answer_results_flag[evaL_answer_results_flag['flag'] == 0]
    final_stat = calculate_eval_stats(eval_answer_results)
    final_stat['model'] = [model] * 7
    total_stat_df.append(final_stat)
total_stat_df = pd.concat(total_stat_df, axis=0)
total_stat_df = total_stat_df.reset_index()
total_stat_df.iloc[:,2:]

,accuracy,semantic_similarity,precision,recall,f1,bertscore_p,bertscore_r,bertscore_f1,pp_id,data,model
0,0.330986,0.686776,0.541453,0.518520,0.518067,0.951160,0.947990,0.949086,76.739437,Total,llama3.1
1,0.161290,0.562231,0.468789,0.464576,0.440761,0.957313,0.953228,0.954823,NaN,Menu,llama3.1
2,0.176471,0.677873,0.326891,0.299241,0.307843,0.888416,0.891969,0.889020,NaN,Dish,llama3.1
3,0.483871,0.770539,0.644355,0.668011,0.648361,0.975466,0.973030,0.974068,NaN,CFI,llama3.1
4,0.434783,0.713796,0.581239,0.533191,0.544857,0.933808,0.942958,0.937943,NaN,PPP,llama3.1
5,0.464286,0.784934,0.575893,0.553231,0.562249,0.986844,0.987402,0.987100,NaN,Hospital,llama3.1
6,0.176471,0.516881,0.548722,0.451691,0.487928,0.925927,0.883383,0.902786,NaN,Flights,llama3.1
7,0.140845,0.520791,0.363483,0.332002,0.331993,0.928183,0.920597,0.923596,76.739437,Total,mistral
8,0.225806,0.543752,0.500047,0.494348,0.463196,0.949282,0.938244,0.942889,NaN,Menu,mistral
9,0.058824,0.548205,0.205882,0.171123,0.181214,0.863060,0.845170,0.852869,NaN,Dish,mistral


In [27]:
total_stat_df.to_csv('answer_performance_table_llama_gemma_mistral.csv', header=True, index=False)

## numeric tags

In [28]:
numeric_pp_tag = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/purposes_category.csv')
numeric_pp_tag.columns = ['pp_id', 'purposes', 'flag']

In [29]:
evaL_answer_results_flag = eval_answer_results.merge(numeric_pp_tag, on='pp_id', how='left', suffixes=('', '_numeric'))

In [30]:
eval_answer_results_text = evaL_answer_results_flag[evaL_answer_results_flag['flag'] == 0]

In [31]:
def eval_answers_numeric(answer_gt_path, answer_preds_llama):
    answer_gt = load_answer_dataset(answer_gt_path)
    answer_gt = pd.DataFrame(answer_gt)
    answer_preds_llama = load_answer_dataset(answer_preds_llama)
    answer_preds_llama = pd.DataFrame(answer_preds_llama)
    answer_compare = answer_gt.merge(answer_preds_llama[['pp_id', 'answer']], on='pp_id', how='left', suffixes=('_gt', '_preds'))
    answer_compare = answer_compare.merge(numeric_pp_tag, on='pp_id', how='left', suffixes=('', '_numeric'))
    answer_compare = answer_compare[answer_compare['flag'] == 1]
    answer_compare['answer_preds'] = pd.to_numeric(answer_compare['answer_preds'], errors='coerce')
    answer_compare['answer_gt'] = pd.to_numeric(answer_compare['answer_gt'], errors='coerce')
    answer_compare['difference'] = (answer_compare['answer_preds'] - answer_compare['answer_gt']).abs()/answer_compare['answer_gt']
    return answer_compare

In [32]:
total_stat_df = []
for model in ['dirty', 'mistral', 'gemma2', 'llama3.1']:
    answer_gt_path = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_gt.json'
    # answer_preds_llama = '/projects/bces/lanl2/LLM4DC/evaluation/answer_1-110_llama3.1.json'
    # model = 'gemma2'
    # model = 'dirty'
    # model = 'llama3.1'
    # model = 'mistral'
    answer_preds = f'/projects/bces/lanl2/LLM4DC/evaluation/answer_1-154_{model}.json'
    # answer_preds = f'/projects/bces/lanl2/LLM4DC/evaluation/answer_1-110_dirty.json'
    eval_answer_results = eval_answers_numeric(answer_gt_path, answer_preds)
    # eval_answer_results.to_csv(f'evaluation/numeric_answer_diff_{model}.csv')
    final_stat = calculate_eval_stats(eval_answer_results[['difference', 'pp_id']])
    final_stat['model'] = [model] * 7
    total_stat_df.append(final_stat)

In [33]:
total_stat_df = pd.concat(total_stat_df, axis=0)

In [34]:
total_stat_df.to_csv('numeric_answer_stats_llamma_mistral_gemma.csv')

In [37]:
dish_results = eval_answer_results_text[eval_answer_results_text['pp_id'] >= 92]
dish_stat_table = dish_results.describe().loc['mean']

In [ ]:
dish_stat_table

In [35]:
eval_answer_results.describe()

,pp_id,answer_gt,answer_preds,flag,difference
count,43.000000,42.000000,42.000000,43.0,42.000000
mean,72.930233,8952.837674,8950.704340,1.0,0.177162
std,52.827840,29816.599232,29817.489095,0.0,0.300400
min,1.000000,0.700000,0.000000,1.0,0.000000
25%,29.000000,3.000000,2.000000,1.0,0.000000
50%,63.000000,9.000000,10.500000,1.0,0.000000
75%,132.500000,75.200000,43.000000,1.0,0.229167
max,150.000000,140400.000000,140400.000000,1.0,1.000000


In [ ]:
ppp_results = eval_answer_results[eval_answer_results['pp_id'] >= 62][eval_answer_results['pp_id'] <=91]

In [38]:
dish_results = eval_answer_results[eval_answer_results['pp_id'] >= 92]

In [ ]:
chi_results = eval_answer_results[eval_answer_results['pp_id'] >= 31][eval_answer_results['pp_id'] <=61]

In [66]:
menu_results = eval_answer_results[eval_answer_results['pp_id'] < 31]

In [ ]:
ppp_results.describe()

In [ ]:
dish_results.describe()

In [ ]:
menu_results.describe()

In [ ]:
chi_results.describe()

In [34]:
eval_answer_results = pd.DataFrame(eval_answer_results)

In [142]:
eval_answer_results['accuracy'].sum()

KeyError: 'accuracy'

In [36]:
eval_answer_results.describe()

,pp_id,answer_gt,answer_preds,flag,difference
count,43.000000,42.000000,42.000000,43.0,42.000000
mean,72.930233,8952.837674,8950.704340,1.0,0.177162
std,52.827840,29816.599232,29817.489095,0.0,0.300400
min,1.000000,0.700000,0.000000,1.0,0.000000
25%,29.000000,3.000000,2.000000,1.0,0.000000
50%,63.000000,9.000000,10.500000,1.0,0.000000
75%,132.500000,75.200000,43.000000,1.0,0.229167
max,150.000000,140400.000000,140400.000000,1.0,1.000000


## answer eval ttest


In [37]:
from scipy import stats

In [38]:
dirty_answer_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/dirty_answer_result.csv')
dirty_answer_result = dirty_answer_result.iloc[:,1:]
gemma2_answer_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/gemma2_answer_result.csv')
gemma2_answer_result = gemma2_answer_result.iloc[:, 1:]
llama_answer_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/llama3.1_answer_result.csv')
llama_answer_result = llama_answer_result.iloc[:,1:]
mistral_answer_result = pd.read_csv('/projects/bces/lanl2/LLM4DC/evaluation/mistral_answer_result.csv')
mistral_answer_result = mistral_answer_result.iloc[:,1:]

In [39]:
answer_result = llama_answer_result.merge(gemma2_answer_result, on='pp_id', how='left', suffixes=('_llama3.1', '_gemma2')).merge(mistral_answer_result, on='pp_id', how='left', suffixes=('', '_mistral'))

In [40]:
answer_result = answer_result.merge(dirty_answer_result, on='pp_id', how='left', suffixes=('', '_dirty'))

In [41]:
answer_result.columns

Index(['accuracy_llama3.1', 'semantic_similarity_llama3.1',
       'precision_llama3.1', 'recall_llama3.1', 'f1_llama3.1',
       'bertscore_p_llama3.1', 'bertscore_r_llama3.1', 'bertscore_f1_llama3.1',
       'pp_id', 'accuracy_gemma2', 'semantic_similarity_gemma2',
       'precision_gemma2', 'recall_gemma2', 'f1_gemma2', 'bertscore_p_gemma2',
       'bertscore_r_gemma2', 'bertscore_f1_gemma2', 'accuracy',
       'semantic_similarity', 'precision', 'recall', 'f1', 'bertscore_p',
       'bertscore_r', 'bertscore_f1', 'accuracy_dirty',
       'semantic_similarity_dirty', 'precision_dirty', 'recall_dirty',
       'f1_dirty', 'bertscore_p_dirty', 'bertscore_r_dirty',
       'bertscore_f1_dirty'],
      dtype='object')

In [42]:
metric_names = ['accuracy','semantic_similarity', 'precision', 'recall', 'f1', 'bertscore_p', 'bertscore_r','bertscore_f1']


In [43]:
ttest_result = []
for m in metric_names:
    model_list = models[:3] + ['dirty'] #['llama', 'gemma2', 'mistral', 'dirty']
    llama_result = stats.ttest_rel(answer_result[f'{m}_dirty'].values, answer_result[f'{m}_llama3.1'].values)
    gemma_result = stats.ttest_rel(answer_result[f'{m}_dirty'].values, answer_result[f'{m}_gemma2'].values)
    # print(answer_result[f'{m}_llama'].values)
    mistral_result = stats.ttest_ind(answer_result[f'{m}_dirty'].values, answer_result[f'{m}'].values)
    ttest_result.append({f'{m}':[llama_result.pvalue.item(), gemma_result.pvalue.item(), mistral_result.pvalue.item()]})
    # print(gemma_result)
    # print(mistral_result)
    # break

In [44]:
def ttest_answer(answer_result):
    ttest_result = []
    for m in metric_names:
        model_list = ['llama', 'gemma2', 'mistral', 'dirty']
        llama_result = stats.ttest_rel(answer_result[f'{m}_dirty'].values, answer_result[f'{m}_llama'].values)
        gemma_result = stats.ttest_rel(answer_result[f'{m}_dirty'].values, answer_result[f'{m}_gemma2'].values)
        # print(answer_result[f'{m}_llama'].values)
        mistral_result = stats.ttest_ind(answer_result[f'{m}_dirty'].values, answer_result[f'{m}'].values)
        ttest_result.append({f'{m}':[llama_result.pvalue.item(), gemma_result.pvalue.item(), mistral_result.pvalue.item()]})
    combined_data = {k: v for d in ttest_result for k, v in d.items()}

    ttest_result = pd.DataFrame(combined_data)
    return ttest_result

In [45]:
combined_data = {k: v for d in ttest_result for k, v in d.items()}

ttest_result = pd.DataFrame(combined_data)


In [46]:
ttest_result['model'] = ['llama3.1', 'gemma2', 'mistral']

In [47]:
ttest_result

,accuracy,semantic_similarity,precision,recall,f1,bertscore_p,bertscore_r,bertscore_f1,model
0,1.356337e-08,8.436960e-11,2.096717e-14,1.066235e-12,8.282881e-15,1.852748e-12,4.726115e-07,1.473374e-10,llama3.1
1,2.663852e-14,4.927548e-16,2.242967e-21,1.817695e-19,6.084980e-22,8.232651e-17,2.417069e-10,1.056240e-14,gemma2
2,4.773402e-01,2.192182e-01,7.251551e-03,3.912747e-02,1.107158e-02,4.154803e-02,6.919708e-01,2.298471e-01,mistral


In [155]:
ppp_results = answer_result[answer_result['pp_id'] >= 62][answer_result['pp_id'] <=91]
ppp_ttest = ttest_answer(ppp_results)

/tmp/ipykernel_4172012/3278763139.py:1: UserWarning: Boolean Series key will be reindexed to match DataFrame index.
  ppp_results = answer_result[answer_result['pp_id'] >= 62][answer_result['pp_id'] <=91]


KeyError: 'accuracy_llama'

In [ ]:
ppp_ttest

In [ ]:
dish_result= answer_result[answer_result['pp_id'] >= 92]
dish_ttest = ttest_answer(dish_result)
dish_ttest

In [ ]:
menu_results = answer_result[answer_result['pp_id'] < 31]
menu_ttest = ttest_answer(menu_results)
menu_ttest

In [ ]:
chi_results = answer_result[answer_result['pp_id'] >= 31][answer_result['pp_id'] <=61]
chi_ttest = ttest_answer(chi_results)
chi_ttest

In [ ]:
ppp_results = answer_result[answer_result['pp_id'] >= 62][answer_result['pp_id'] <=91]
ppp_ttest = ttest_answer(ppp_results)
ppp_ttest